# MLflow 实验：记录两次模型训练（修正版）

依次运行每个 cell（Shift+Enter）。和老师原版的区别见分步指南 PDF。

## 第一部分：先不用 MLflow，训练两个模型

In [1]:
# === Model Training and Experimentation ===
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [2]:
# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [3]:
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [4]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)


Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


## 第二部分：加入 MLflow，在本地记录实验

In [5]:
import mlflow
import mlflow.sklearn

In [6]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='file:c:/Users/ZCQ/Desktop/DEVOPS AND MLOPS/MLOPS/mlflow-lab/mlruns/1', creation_time=1792120235511, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792120235511, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

In [9]:
# Train a model, log parameters, metrics and the model to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name):
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        model_info = mlflow.sklearn.log_model(model, name="model", serialization_format="cloudpickle")

        print(f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")
        return model_info.model_uri

In [10]:
print("Training and Logging Model 1...")
uri_1 = train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

2026/10/16 11:27:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Training and Logging Model 1...
Logged RandomForest model with n_estimators=20, max_depth=5, accuracy=0.9389


In [11]:
print("\nTraining and Logging Model 2...")
uri_2 = train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2")


Training and Logging Model 2...


2026/10/16 11:27:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged RandomForest model with n_estimators=100, max_depth=10, accuracy=0.9722


In [12]:
print("\nTo view the results, run the following command in your terminal:")
print("mlflow ui")
print("Then navigate to http://127.0.0.1:5000 to explore the experiment results.")


To view the results, run the following command in your terminal:
mlflow ui
Then navigate to http://127.0.0.1:5000 to explore the experiment results.


## 第三部分：把模型注册到 Model Registry，再加载出来

In [16]:
mv = mlflow.register_model(uri_2, "digits-rf")
print("Registered:", mv.name, "version", mv.version)

Registered: digits-rf version 2


Registered model 'digits-rf' already exists. Creating a new version of this model...
Created version '2' of model 'digits-rf'.


In [15]:
loaded_model = mlflow.sklearn.load_model(f"models:/digits-rf/{mv.version}")
print("Predictions:", loaded_model.predict(X_test[:10]))
print("Real labels:", y_test[:10])

Predictions: [6 9 3 7 2 1 5 2 5 2]
Real labels: [6 9 3 7 2 1 5 2 5 2]


## 第四部分：测试部署好的模型

先在终端运行 `mlflow models serve ...`（见指南步骤 9），看到服务启动后再运行下面这个 cell。

In [17]:
import requests

response = requests.post(
    "http://127.0.0.1:5003/invocations",
    json={"inputs": X_test[:5].tolist()},
)
print("Server says:", response.json())
print("Real labels:", y_test[:5])

Server says: {'predictions': [6, 9, 3, 7, 2]}
Real labels: [6 9 3 7 2]


## 第五部分：连接 DagsHub，把实验记录到远程

把下面这行换成 DagsHub 网页上复制的代码（见指南步骤 11）。

In [18]:
# get this code from dagshub
import dagshub
dagshub.init(repo_owner='ZCQ-wqqq', repo_name='mlflow-lab', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\ZCQ\Desktop\DEVOPS AND MLOPS\MLOPS\mlflow-lab\.venv\Lib\site-packages\rich\live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=052d00f4-4151-4a93-9e31-3366b20802d1&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=b2642a11afdd0529e12a75ebcb647a63c402268d6c2dd04020621aad4785656b




Accessing as ZCQ-wqqq

Initialized MLflow to track repo "ZCQ-wqqq/mlflow-lab"

Repository ZCQ-wqqq/mlflow-lab initialized!

In [19]:
mlflow.set_experiment(experiment_name)
uri_1 = train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")
uri_2 = train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2")

2026/10/16 13:34:40 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_V1.0.1' does not exist. Creating a new experiment.
2026/10/16 13:34:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged RandomForest model with n_estimators=20, max_depth=5, accuracy=0.9389
🏃 View run random_forest_model_run_1 at: https://dagshub.com/ZCQ-wqqq/mlflow-lab.mlflow/#/experiments/0/runs/f90dea17e37846f2b964ecd96b87bdcb
🧪 View experiment at: https://dagshub.com/ZCQ-wqqq/mlflow-lab.mlflow/#/experiments/0


2026/10/16 13:34:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged RandomForest model with n_estimators=100, max_depth=10, accuracy=0.9722
🏃 View run random_forest_model_run_V1.0.2 at: https://dagshub.com/ZCQ-wqqq/mlflow-lab.mlflow/#/experiments/0/runs/f255402abbcc417b9b151a7d0d0a2ecb
🧪 View experiment at: https://dagshub.com/ZCQ-wqqq/mlflow-lab.mlflow/#/experiments/0
